## Rehash from geocode_cache_t


## Pathing

In [1]:
import pandas as pd
import os
import hashlib
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

input_path = Path(
    "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint3_geocoded.csv"
)

output_path = input_path.with_name("checkpoint5_hashed.csv")
mapping_path = input_path.with_name("name_mapping.csv")
# ------------------------------------------------------------
# Load geocoded checkpoint
# ------------------------------------------------------------

df = pd.read_csv(input_path)

print(f"Loaded rows: {len(df):,}")
print(f"Input file: {input_path}")


Loaded rows: 428,527
Input file: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint3_geocoded.csv


/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_25902/1398345701.py:20: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


## Hashing

In [2]:


# ------------------------------------------------------------
# Hash function
# ------------------------------------------------------------

def generate_id(name, dob):
    if pd.isna(name) or pd.isna(dob):
        return ""

    combined = f"{str(name).strip()}_{str(dob).strip()}"
    return hashlib.sha256(combined.encode()).hexdigest()


# ------------------------------------------------------------
# Create person_id
# ------------------------------------------------------------

required_cols = ["Name", "DOB", "Charges"]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["person_id"] = df.apply(
    lambda row: generate_id(row["Name"], row["DOB"])
    if pd.notna(row["Charges"]) and str(row["Charges"]).strip() != ""
    else "",
    axis=1
)

# ------------------------------------------------------------
# Save mapping file before dropping names
# ------------------------------------------------------------

name_mapping = (
    df[df["person_id"] != ""][["person_id", "Name", "DOB"]]
    .drop_duplicates()
)

name_mapping.to_csv(mapping_path, index=False)

print(f"Saved name mapping rows: {len(name_mapping):,}")
print(f"Mapping file: {mapping_path}")

# ------------------------------------------------------------
# Drop Name column for anonymized checkpoint
# ------------------------------------------------------------

df_hashed = df.drop(columns=["Name"])

# ------------------------------------------------------------
# Save hashed checkpoint
# ------------------------------------------------------------

df_hashed.to_csv(output_path, index=False)

print(f"Saved hashed checkpoint rows: {len(df_hashed):,}")
print(f"Output file: {output_path}")

Saved name mapping rows: 4,287
Mapping file: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/name_mapping.csv
Saved hashed checkpoint rows: 428,527
Output file: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint5_hashed.csv
